# Notebook 05 — Perfil Agregado por Parlamentar

**Sprint 3 — Lei e Política**

Perfil descritivo (não-supervisionado) de cada deputado por tema, consumido pelo frontend
da Sprint 4. Para cada par (parlamentar × tema) calculamos o percentual de votos
favoráveis e classificamos a postura.

## Pipeline

1. Junção `votos` × `votacoes` × `proposicoes` (tema_cidadao) × `parlamentares`
2. Agregação: `pct_favoravel` e `total_votacoes` por parlamentar × tema (mín. 3 votos decisivos)
3. Classificação de `postura_geral`: ≥65% → favorável · ≤35% → contrário · senão neutro
4. Gravação em `perfil_parlamentar`

In [1]:
import sys
sys.path.insert(0, '..')

import logging

import pandas as pd

from src.db import buscar_todos, upsert_perfil

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
log = logging.getLogger('05_perfil')
print('Módulos carregados.')

Módulos carregados.


## 1. Montagem da base de votos

Cada linha é um voto decisivo (`favoravel`/`contrario`) de um deputado, ligado ao tema da
proposição. Juntamos `votos` → `votacoes` → `proposicoes` (`tema_cidadao`) →
`parlamentares`. Votos sem tema (votação sem proposição linkada) e abstenções saem do
cálculo de percentual.

In [2]:
# Carrega e junta as tabelas (escopo: Câmara — só há votos nominais da Câmara)
votos = pd.DataFrame(buscar_todos('votos', 'votacao_id,parlamentar_id,voto'))
votacoes = pd.DataFrame(buscar_todos('votacoes', 'id,proposicao_id'))
proposicoes = pd.DataFrame(buscar_todos('proposicoes', 'id,tema_cidadao'))
parlamentares = pd.DataFrame(buscar_todos('parlamentares', 'id,nome,casa'))

print(f'votos={len(votos)}  votacoes={len(votacoes)}  '
      f'proposicoes={len(proposicoes)}  parlamentares={len(parlamentares)}')

# Chaves de junção como inteiro anulável (proposicao_id vem como object por conter nulos)
for d, col in [(votos, 'votacao_id'), (votos, 'parlamentar_id'),
               (votacoes, 'id'), (votacoes, 'proposicao_id'),
               (proposicoes, 'id'), (parlamentares, 'id')]:
    d[col] = pd.to_numeric(d[col], errors='coerce').astype('Int64')

df = votos.merge(votacoes, left_on='votacao_id', right_on='id', suffixes=('', '_vt'))
df = df.merge(proposicoes, left_on='proposicao_id', right_on='id', suffixes=('', '_pr'))
df = df.merge(parlamentares, left_on='parlamentar_id', right_on='id', suffixes=('', '_pl'))

# Apenas Câmara, com tema definido e voto decisivo
df = df[df['casa'] == 'camara']
df = df[df['tema_cidadao'].notna()]
df = df[df['voto'].isin(['favoravel', 'contrario'])].copy()

df['fav'] = (df['voto'] == 'favoravel').astype(int)

print(f'\nVotos decisivos com tema: {len(df)}')
print(f'Parlamentares distintos: {df["parlamentar_id"].nunique()}')
print(f'Temas distintos: {df["tema_cidadao"].nunique()}')

2026-06-23 23:18:11,272 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=0&limit=1000 "HTTP/2 200 OK"
2026-06-23 23:18:11,582 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=1000&limit=1000 "HTTP/2 200 OK"
2026-06-23 23:18:11,845 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=2000&limit=1000 "HTTP/2 200 OK"
2026-06-23 23:18:12,108 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=3000&limit=1000 "HTTP/2 200 OK"
2026-06-23 23:18:12,498 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=4000&limit=1000 "HTTP/2 200 OK"
2026-06-23 23:18:12,813 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.

votos=55570  votacoes=137  proposicoes=22041  parlamentares=726

Votos decisivos com tema: 16798
Parlamentares distintos: 574
Temas distintos: 4


## 2. Agregação por parlamentar × tema

Para cada par (parlamentar, tema): `total_votacoes` = nº de votos decisivos e
`pct_favoravel` = % de favoráveis. Guardamos só pares com **≥ 3 votos** (evita
percentuais ruidosos de 1–2 votos). A postura segue os cortes do schema:
≥ 65% favorável · ≤ 35% contrário · senão neutro.

In [3]:
MIN_VOTOS = 3

agg = (
    df.groupby(['parlamentar_id', 'tema_cidadao'])
      .agg(total_votacoes=('fav', 'size'), favoraveis=('fav', 'sum'))
      .reset_index()
)
agg = agg[agg['total_votacoes'] >= MIN_VOTOS].copy()
agg['pct_favoravel'] = (agg['favoraveis'] / agg['total_votacoes'] * 100).round(2)

def classificar(pct):
    if pct >= 65:
        return 'favoravel'
    if pct <= 35:
        return 'contrario'
    return 'neutro'

agg['postura_geral'] = agg['pct_favoravel'].map(classificar)

print(f'Perfis (parlamentar × tema) com >= {MIN_VOTOS} votos: {len(agg)}')
print('\nDistribuição por postura_geral:')
print(agg['postura_geral'].value_counts())
print('\nAmostra:')
print(agg.sort_values('total_votacoes', ascending=False).head(10).to_string(index=False))

Perfis (parlamentar × tema) com >= 3 votos: 1062

Distribuição por postura_geral:
postura_geral
neutro       625
favoravel    291
contrario    146
Name: count, dtype: int64

Amostra:
 parlamentar_id                           tema_cidadao  total_votacoes  favoraveis  pct_favoravel postura_geral
            281 Políticas Públicas e Programas Sociais              33          19          57.58        neutro
            248 Políticas Públicas e Programas Sociais              33          20          60.61        neutro
            254 Políticas Públicas e Programas Sociais              33          18          54.55        neutro
            311 Políticas Públicas e Programas Sociais              33          20          60.61        neutro
            376 Políticas Públicas e Programas Sociais              33          19          57.58        neutro
             56 Políticas Públicas e Programas Sociais              33          18          54.55        neutro
            582 Políticas Pública

## 3. Gravar em `perfil_parlamentar` e validar

`upsert_perfil` faz upsert por (`parlamentar_id`, `tema_cidadao`), então reexecutar o
notebook atualiza os perfis sem duplicar.

In [4]:
registros = [
    {
        'parlamentar_id': int(r['parlamentar_id']),
        'tema_cidadao': r['tema_cidadao'],
        'pct_favoravel': float(r['pct_favoravel']),
        'total_votacoes': int(r['total_votacoes']),
        'postura_geral': r['postura_geral'],
    }
    for _, r in agg.iterrows()
]
total = upsert_perfil(registros)
print(f'{total} perfis gravados em perfil_parlamentar.')

2026-06-23 23:18:51,109 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?on_conflict=parlamentar_id%2Ctema_cidadao&columns=%22parlamentar_id%22%2C%22pct_favoravel%22%2C%22tema_cidadao%22%2C%22total_votacoes%22%2C%22postura_geral%22 "HTTP/2 201 Created"
2026-06-23 23:18:51,474 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?on_conflict=parlamentar_id%2Ctema_cidadao&columns=%22parlamentar_id%22%2C%22pct_favoravel%22%2C%22tema_cidadao%22%2C%22total_votacoes%22%2C%22postura_geral%22 "HTTP/2 201 Created"
2026-06-23 23:18:51,881 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?on_conflict=parlamentar_id%2Ctema_cidadao&columns=%22parlamentar_id%22%2C%22pct_favoravel%22%2C%22tema_cidadao%22%2C%22total_votacoes%22%2C%22postura_geral%22 "HTTP/2 201 Created"
2026-06-23 23:18:51,886 [INFO] upsert perfil_parlamentar: 1062 registros


1062 perfis gravados em perfil_parlamentar.


In [5]:
# Sanidade: relê o que foi gravado
check = pd.DataFrame(buscar_todos(
    'perfil_parlamentar',
    'parlamentar_id,tema_cidadao,pct_favoravel,total_votacoes,postura_geral',
))
print(f'Perfis em perfil_parlamentar: {len(check)}')

if check.empty:
    print('\n⚠️  Tabela vazia. Causa provável: as proposições ligadas às votações ainda '
          'não têm `tema_cidadao` (rode o notebook 03 / Sprint 2 e grave os temas). '
          'Veja o print "Votos decisivos com tema" da célula 1: se for 0, é isso.')
else:
    print('\nDistribuição por postura_geral:')
    print(check['postura_geral'].value_counts())
    print('\nAmostra:')
    print(check.head(10).to_string(index=False))

2026-06-23 23:18:59,612 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?select=parlamentar_id%2Ctema_cidadao%2Cpct_favoravel%2Ctotal_votacoes%2Cpostura_geral&offset=0&limit=1000 "HTTP/2 200 OK"
2026-06-23 23:18:59,926 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?select=parlamentar_id%2Ctema_cidadao%2Cpct_favoravel%2Ctotal_votacoes%2Cpostura_geral&offset=1000&limit=1000 "HTTP/2 200 OK"


Perfis em perfil_parlamentar: 1062

Distribuição por postura_geral:
postura_geral
neutro       625
favoravel    291
contrario    146
Name: count, dtype: int64

Amostra:
 parlamentar_id                           tema_cidadao  pct_favoravel  total_votacoes postura_geral
              6  Educação, Trânsito e Direitos Sociais          42.86               7        neutro
              6 Políticas Públicas e Programas Sociais          73.91              23     favoravel
              7  Educação, Trânsito e Direitos Sociais          28.57               7     contrario
              7 Políticas Públicas e Programas Sociais          58.06              31        neutro
              8  Educação, Trânsito e Direitos Sociais          71.43               7     favoravel
              8 Políticas Públicas e Programas Sociais          62.96              27        neutro
              9  Educação, Trânsito e Direitos Sociais          57.14               7        neutro
              9 Políticas Públi

In [6]:
print('votações c/ proposicao_id :', votacoes['proposicao_id'].notna().sum(), 'de', len(votacoes))
print('proposições c/ tema_cidadao:', proposicoes['tema_cidadao'].notna().sum(), 'de', len(proposicoes))

votações c/ proposicao_id : 42 de 137
proposições c/ tema_cidadao: 22040 de 22041
